# 🔥 OrbitFire AI — Análise de Interpretabilidade com SHAP

**GAIE — Generative AI for Engineering | FIAP Global Solution**

**Squad:** Sentinela Corp — [github.com/sentinela-corp/sentinela-app-gaie](https://github.com/sentinela-corp/sentinela-app-gaie)

**Autores:** Deivison Pertel (RM 550803) · Eduardo Akira Murata (RM 98713) · Wesley Souza de Oliveira (RM 97874)

Este notebook aprofunda a análise de interpretabilidade do modelo de previsão de risco de queimadas usando SHAP (SHapley Additive exPlanations).

---
**ODS 13** — Ação Climática | Satélites & IA para monitoramento ambiental


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import shap
from sklearn.inspection import PartialDependenceDisplay

shap.initjs()
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Bibliotecas carregadas.')

## 1. Carregamento do Modelo e Dataset

In [ ]:
modelo = joblib.load('outputs/modelo_orbitfire.pkl')

with open('outputs/model_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

features = meta['features']
classes  = meta['classes']

print(f'Modelo carregado: {meta["melhor_modelo"]}')
print(f'Features ({len(features)}): {features}')
print(f'Classes: {classes}')

In [ ]:
df = pd.read_csv('outputs/dataset_orbitfire.csv')
print(f'Dataset: {df.shape}')
print(df['risco_queimada'].value_counts())
df.head()

## 2. Pré-processamento (replicando pipeline)

In [ ]:
def engenharia_features(df):
    df = df.copy()
    df['indice_aridez']        = df['temperatura'] / (df['umidade_relativa'] + 1)
    df['potencial_propagacao'] = df['velocidade_vento'] * (1 - df['umidade_relativa'] / 100)
    df['deficit_hidrico']      = df['dias_sem_chuva'] * (1 - np.clip(df['precipitacao_24h'] / 10, 0, 1))
    df['estacao_seca']         = df['mes'].apply(lambda m: 1 if 5 <= m <= 10 else 0)
    df['ndvi_temp']            = df['ndvi'] * df['temperatura']
    return df

df_eng = engenharia_features(df)
X = df_eng[features]
y = df['risco_queimada'].astype(str)

# Amostra para SHAP (computacionalmente intensivo)
X_sample = X.sample(500, random_state=42)
print(f'Amostra SHAP: {X_sample.shape}')

## 3. Computação dos Valores SHAP

In [ ]:
inner_model = modelo.named_steps['model']
scaler      = modelo.named_steps['scaler']
X_scaled    = scaler.transform(X_sample)

explainer = shap.TreeExplainer(inner_model)
shap_values = explainer.shap_values(X_scaled)

print(f'Tipo: {type(shap_values)}')
if isinstance(shap_values, list):
    print(f'Formato por classe: {[sv.shape for sv in shap_values]}')
else:
    print(f'Formato: {shap_values.shape}')

## 4. SHAP Summary Plot — Importância Global

In [ ]:
# Classe 'alto' (índice 0 na ordem alfabética)
idx_alto = list(inner_model.classes_).index('alto')

sv_alto = shap_values[idx_alto] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(
    sv_alto, X_sample,
    feature_names=features,
    plot_type='bar',
    color='#E25822',
    show=False
)
plt.title('SHAP — Importância Global das Variáveis (Risco Alto)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Salvo: outputs/shap_summary_bar.png')

## 5. SHAP Beeswarm Plot — Distribuição de Impactos

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_alto, X_sample,
    feature_names=features,
    show=False
)
plt.title('SHAP Beeswarm — Distribuição de Impactos (Risco Alto)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Salvo: outputs/shap_beeswarm.png')

## 6. SHAP Waterfall — Explicação Individual

In [ ]:
# Pegar uma amostra de risco alto
df_alto = df_eng[y == 'alto'].sample(1, random_state=7)
X_alto  = df_alto[features]
X_alto_scaled = scaler.transform(X_alto)

sv_ind = explainer.shap_values(X_alto_scaled)
sv_ind_alto = sv_ind[idx_alto][0] if isinstance(sv_ind, list) else sv_ind[0]

fig, ax = plt.subplots(figsize=(10, 6))
contrib = pd.Series(sv_ind_alto, index=features).sort_values()
cores = ['#27AE60' if v < 0 else '#C0392B' for v in contrib]
contrib.plot(kind='barh', ax=ax, color=cores, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('SHAP Waterfall — Explicação Individual (Risco Alto)', fontweight='bold')
ax.set_xlabel('Contribuição SHAP')

red_p   = mpatches.Patch(color='#C0392B', label='Aumenta risco')
green_p = mpatches.Patch(color='#27AE60', label='Reduz risco')
ax.legend(handles=[red_p, green_p])
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/shap_waterfall_individual.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nValores da amostra:')
print(df_alto[['temperatura','umidade_relativa','velocidade_vento','dias_sem_chuva']].T)

## 7. Análise por Classe — Comparativo SHAP

In [ ]:
if isinstance(shap_values, list) and len(shap_values) == 3:
    nomes_classes = inner_model.classes_
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    cores_classe = {'alto': '#C0392B', 'médio': '#E67E22', 'baixo': '#27AE60'}

    for ax, (sv, cls) in zip(axes, zip(shap_values, nomes_classes)):
        importances = np.abs(sv).mean(axis=0)
        top_idx = np.argsort(importances)[-8:]
        top_feats = [features[i] for i in top_idx]
        top_vals  = importances[top_idx]
        ax.barh(top_feats, top_vals, color=cores_classe.get(cls, '#555'), alpha=0.85, edgecolor='white')
        ax.set_title(f'Classe: {cls}', fontweight='bold', color=cores_classe.get(cls, '#555'))
        ax.set_xlabel('|SHAP| médio')
        ax.grid(axis='x', alpha=0.3)

    fig.suptitle('SHAP — Top Features por Classe de Risco', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('outputs/shap_por_classe.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Salvo: outputs/shap_por_classe.png')
else:
    print('Modelo não multiclasse — gráfico por classe não aplicável.')

## 8. Dependência Parcial — Temperatura × Umidade

In [ ]:
# Insight: como temperatura e umidade interagem no risco
from sklearn.inspection import partial_dependence

fig, ax = plt.subplots(figsize=(9, 5))

# Gradiente de risco manual via modelo
temps = np.linspace(10, 45, 40)
umids = [20, 40, 60, 80]

for umid in umids:
    rows = []
    for t in temps:
        rows.append({
            'temperatura': t, 'umidade_relativa': umid,
            'velocidade_vento': 30, 'precipitacao_24h': 1,
            'ndvi': 0.3, 'dias_sem_chuva': 40,
            'altitude': 500, 'mes': 8, 'historico_focos': 5
        })
    df_line = engenharia_features(pd.DataFrame(rows))[features]
    probas  = modelo.predict_proba(df_line)
    idx_alto_col = list(modelo.classes_).index('alto')
    ax.plot(temps, probas[:, idx_alto_col], label=f'Umidade {umid}%', linewidth=2)

ax.set_xlabel('Temperatura (°C)', fontweight='bold')
ax.set_ylabel('P(Risco Alto)', fontweight='bold')
ax.set_title('Dependência Parcial — P(Alto) vs Temperatura por Umidade', fontweight='bold')
ax.legend(title='Umidade Relativa')
ax.grid(alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('outputs/partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Salvo: outputs/partial_dependence.png')

## 9. Conclusões

### Achados principais da análise SHAP:

1. **Umidade Relativa** é a variável com maior impacto negativo no risco — valores baixos de umidade aumentam dramaticamente a probabilidade de risco alto.
2. **Temperatura** contribui positivamente para o risco, especialmente acima de 35°C.
3. **Déficit Hídrico** (feature derivada) captura bem a combinação de dias sem chuva + precipitação recente, sendo um dos mais importantes.
4. **Índice de Aridez** (temperatura/umidade) resume de forma eficaz as condições atmosféricas de risco.
5. **NDVI** tem impacto moderado — vegetação mais seca (NDVI baixo) eleva o risco, como esperado.
6. **Velocidade do vento** amplifica o risco ao aumentar o potencial de propagação do fogo.
7. **Sazonalidade** (estação seca / mês) confirma padrão já conhecido: risco maior de maio a outubro.

### Implicações para o sistema de alertas:
- Threshold de alerta recomendado: P(alto) > 0.60
- Combinação crítica: Temperatura > 38°C + Umidade < 15% + Vento > 40km/h + dias_sem_chuva > 30

---
*OrbitFire AI · Sentinela Corp · github.com/sentinela-corp/sentinela-app-gaie · GAIE | ODS 13 — Ação Climática*